In [ ]:
!pip install -U transformers sentence-transformers chromadb accelerate "pillow<11.0"
import os
import shutil
import zipfile
import torch
import chromadb
from PIL import Image
from transformers import ViltProcessor, ViltForQuestionAnswering
from transformers import pipeline
from sentence_transformers import SentenceTransformer

INPUT_PATH = "/kaggle/input/knowledge-dataset"
WORK_DIR = "/kaggle/working/db_restored"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def find_db_source(search_path):
    for root, dirs, files in os.walk(search_path):
        if "chroma.sqlite3" in files:
            return root
    return None

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

DB_PATH = os.path.join(WORK_DIR, "chroma_db_final")
IMG_PATH = ""

if os.path.isdir(INPUT_PATH):
    print(f"Input is a DIRECTORY: {INPUT_PATH}")
    db_source = find_db_source(INPUT_PATH)

    if db_source:
        print(f"  -> Found Database at: {db_source}")
        print(f"  -> Copying to writable storage...")
        shutil.copytree(db_source, DB_PATH)
    else:
        print(f"Error: Could not find 'chroma.sqlite3' anywhere in {INPUT_PATH}")
        print("   Run '!find /kaggle/input -name chroma.sqlite3' to debug.")
    possible_img_path = os.path.join(INPUT_PATH, "extracted_images")
    if os.path.exists(possible_img_path):
        IMG_PATH = possible_img_path
        print(f"  -> Found images at: {IMG_PATH}")
    else:
        IMG_PATH = INPUT_PATH
        print(f"explicit 'extracted_images' folder not found. Looking in root: {IMG_PATH}")

elif zipfile.is_zipfile(INPUT_PATH):
    print(f"Input is a ZIP file. Extracting...")
    with zipfile.ZipFile(INPUT_PATH, 'r') as zip_ref:
        zip_ref.extractall(WORK_DIR)
    db_source = find_db_source(WORK_DIR)
    if db_source:
      DB_PATH = db_source

    IMG_PATH = os.path.join(WORK_DIR, "extracted_images")
    if not os.path.exists(IMG_PATH):
        IMG_PATH = WORK_DIR
    print("Extraction complete.")
else:
    print(f"Error: {INPUT_PATH} not found.")
try:
    if os.path.exists(DB_PATH):
        client = chromadb.PersistentClient(path=DB_PATH)
        col_names = [c.name for c in client.list_collections()]
        print(f"Database connected. Collections found: {col_names}")
        if "pdf_knowledge_base" in col_names:
            collection = client.get_collection("pdf_knowledge_base")
        elif len(col_names) > 0:
            print(f"'pdf_knowledge_base' not found. Defaulting to '{col_names[0]}'")
            collection = client.get_collection(col_names[0])
        else:
            print("Database is empty (no collections).")
            collection = None

        if collection:
            print(f"Total Documents: {collection.count()}")
    else:
        print(f"Database path invalid.")
        collection = None
except Exception as e:
    print(f"Critical Error loading database: {e}")
    collection = None

import time

def get_query_embedding(query_text):
    embed_model.to(device)

    with torch.no_grad():
        query_embedding = embed_model.encode(query_text).tolist()
    embed_model.to("cpu")
    torch.cuda.empty_cache()

    return query_embedding

def ask_bot(question):
    start_total = time.time()

    print(f"\n{'='*60}")
    print(f" QUESTION: {question}")
    print(f"{'='*60}")

    if collection is None:
        print("Database not ready.")
        return

    # 2. Retrieve Documents (Measure Latency)
    print(" Retrieving context with EmbeddingGemma...")
    start_retrieval = time.time()

    query_embed = get_query_embedding(question)
    results = collection.query(query_embeddings=[query_embed], n_results=3)

    end_retrieval = time.time()
    print(f"  Retrieval Latency: {end_retrieval - start_retrieval:.4f}s")

    if not results['documents'] or not results['documents'][0]:
        print("No information found.")
        return
    print("\n---Text Expert (RoBERTa) ---")
    context_text = " ".join(results['documents'][0])
    context_text = context_text[:3000]

    start_text = time.time()
    try:
        text_ans = qa_pipeline(question=question, context=context_text)
        print(f"Answer: {text_ans['answer']}")
        print(f"   (Confidence: {text_ans['score']:.2f})")
    except Exception as e:
        print(f"   Error: {e}")
    end_text = time.time()
    print(f"   ↳ Text Processing Latency: {end_text - start_text:.4f}s")

    print("\n--- Visual Expert (ViLT) ---")

    start_vision = time.time()
    metadatas = results['metadatas'][0]
    found_images = False

    for meta in metadatas:
        img_str = meta.get('images', '') or meta.get('image_paths', '')

        if img_str:
            images = img_str.split(',')
            for img_file in images:
                img_file = img_file.strip()
                full_path = os.path.join(IMG_PATH, os.path.basename(img_file))

                if os.path.exists(full_path):
                    found_images = True
                    try:
                        image = Image.open(full_path).convert("RGB")
                        encoding = vilt_processor(image, question, return_tensors="pt")
                        encoding = {k: v.to(device) for k, v in encoding.items()}

                        with torch.no_grad():
                            outputs = vilt_model(**encoding)

                        logits = outputs.logits
                        idx = logits.argmax(-1).item()
                        answer = vilt_model.config.id2label[idx]

                        print(f"[Image: {img_file}] -> ViLT says: '{answer}'")

                    except Exception as e:
                        print(f"   Error reading {img_file}: {e}")
                else:
                    pass

    if not found_images:
        print("   No relevant images found in retrieved context.")
    end_vision = time.time()
    print(f"Visual Processing Latency: {end_vision - start_vision:.4f}s")
    end_total = time.time()
    print(f"\nTOTAL QUERY LATENCY: {end_total - start_total:.4f}s")

if __name__ == "__main__":
    print("\nSYSTEM READY.")
    while True:
        try:
            user_input = input("\nEnter your question (or 'q' to exit): ")
            if user_input.lower() in ['q', 'quit', 'exit']:
                break

            if user_input.strip():
                ask_bot(user_input)
        except KeyboardInterrupt:
            print("\nExiting...")
            break